### rag pipeline -data ingestion to vector db pipeline


In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path




C:\Users\Kishan Prajapati\AppData\Local\Temp\ipykernel_13760\1700143242.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
d:\learning projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## read al pdf inside dir

def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# process all pdfs in data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: CODINGGUIDELINESFORCTSYS.pdf
 Loaded 5 pages

Total documents loaded: 5


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m59', 'creator': 'PyPDF', 'creationdate': '2017-05-22T12:39:05+05:30', 'moddate': '2017-05-22T12:39:05+05:30', 'source': '..\\data\\pdf_files\\CODINGGUIDELINESFORCTSYS.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'CODINGGUIDELINESFORCTSYS.pdf', 'file_type': 'pdf'}, page_content='CODING GUIDELINES FOR CTSYS  \n  \n  \n Throughout the application we should follow the following coding conventions::  \n  \n Folder Naming :  \n Folder Names should be Pascal cased.  \n Ex:- ErrorLogs, Reports.  \n  \n File Naming :  \n File Names should be Pascal cased. In the Model and Controller their  \n names should be followed by a Model/Controller postfix.  \n Ex:- LoginModel.cfc, LoginController.cfc.  \n *Use the pre existing file namings.  \n  \n Variable Naming :  \n Variable Names should be Camel cased.  \n Variables should be written with proper scopes as prefix like APPLICATION, SESSION).\n Ex:- userInfo( camelCased )  \n  \n Funct

In [4]:
### text splitting into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

chunks=split_documents(all_pdf_documents)
chunks


Split 5 documents into 12 chunks

Example chunk:
Content: CODING GUIDELINES FOR CTSYS  
  
  
 Throughout the application we should follow the following coding conventions::  
  
 Folder Naming :  
 Folder Names should be Pascal cased.  
 Ex:- ErrorLogs, Rep...
Metadata: {'producer': 'Skia/PDF m59', 'creator': 'PyPDF', 'creationdate': '2017-05-22T12:39:05+05:30', 'moddate': '2017-05-22T12:39:05+05:30', 'source': '..\\data\\pdf_files\\CODINGGUIDELINESFORCTSYS.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'CODINGGUIDELINESFORCTSYS.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m59', 'creator': 'PyPDF', 'creationdate': '2017-05-22T12:39:05+05:30', 'moddate': '2017-05-22T12:39:05+05:30', 'source': '..\\data\\pdf_files\\CODINGGUIDELINESFORCTSYS.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'CODINGGUIDELINESFORCTSYS.pdf', 'file_type': 'pdf'}, page_content='CODING GUIDELINES FOR CTSYS  \n  \n  \n Throughout the application we should follow the following coding conventions::  \n  \n Folder Naming :  \n Folder Names should be Pascal cased.  \n Ex:- ErrorLogs, Reports.  \n  \n File Naming :  \n File Names should be Pascal cased. In the Model and Controller their  \n names should be followed by a Model/Controller postfix.  \n Ex:- LoginModel.cfc, LoginController.cfc.  \n *Use the pre existing file namings.  \n  \n Variable Naming :  \n Variable Names should be Camel cased.  \n Variables should be written with proper scopes as prefix like APPLICATION, SESSION).\n Ex:- userInfo( camelCased )  \n  \n Funct

### embedding and vector store db

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity



In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformers model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded sucessfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    # def get_embedding_dimension(self) -> int:
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #     return self.model.get_embedding_dimension()

## initalize the embedidng manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2868.23it/s]


Model loaded sucessfully. Embedding dimension: 384


### vector store

In [7]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist th vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing douments in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initlizing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embeddings) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embeddings.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing douments in collection: 36


In [8]:
### convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##Store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 12 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.29it/s]

Generated embeddings with shape: (12, 384)
Adding 12 documents to vector store...
Successfully added 12 documents to vector store
Total documents in collection: 48


### Retriever Pipeline from vector store

In [30]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore, embedding_manager)
rag_retriever.retrieve(query = "Naming Convention for Stored Procedures")

Retrieving documents for query: 'Naming Convention for Stored Procedures
Top K: 5, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.21it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_41c66d7f_2',
  'content': 'Naming Convention for Stored Procedures :  \n BACKOFFICE_section_SPNAME  \n COMMON_SPName  \n  \n Function Header :  \n /*-------------------------------------------------------------------------------------------------------------------------  \n Function Name:  \n Description: This function checks whether the emailId exists or not in the datasource.  \n Arguments: 1. emailId-It describes the emailId.  \n        2.id- Its the user id.  \n Return Type: boolean.  \n ---------------------------------------------------------------------------------------------------------------------------*/  \n  \n Inline Comments :  \n Every critical part of code should be properly commented. It’s better to even give inline comments at the  \n start of a block of code accomplishing a particular functionality.  \n Any file should not be having unused piece of code as commented. If needed then we can  \n comment with proper reasoning.  \n  \n Use of Hint :',
  'meta

In [31]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retrieve the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    # generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}
        
        Answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [32]:
answer=rag_simple("Naming Conventions", rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Naming Conventions
Top K: 3, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.10it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Naming Conventions (CTSYS)**  

| Category | Convention | Example |
|----------|------------|---------|
| **Folder** | PascalCase | `ErrorLogs`, `Reports` |
| **File** | PascalCase; Models/Controllers end with `Model`/`Controller` | `LoginModel.cfc`, `LoginController.cfc` |
| **Variable** | camelCase with scope prefix | `APPLICATION.userInfo`, `SESSION.cartItems` |
| **Function** | PascalCase, descriptive of action | `GetUserDetails`, `UpdateOrderStatus` |
| **Whitespace** | Even spaces around logical operators and syntax | `if (a == b) { … }` |

Follow these rules consistently across the application.


Enhanced RAG Pipeline Features

In [49]:
# Enhanced RAG Pipeline Features
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare cotext amd sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.
        
        \nContext:\n{context}
        
        \n\nQuesiton: {query}\n\n
        
        Answer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Naming Convention for Stored Procedures", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence", result['confidence'])
print("Context Preview", result['context'][:300])

Retrieving documents for query: 'Naming Convention for Stored Procedures
Top K: 3, score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.87it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **Naming Convention for Stored Procedures**

| Context | Convention |
|---------|------------|
| Back‑office section | `BACKOFFICE_<section>_<SPName>` |
| Common/shared procedures | `COMMON_<SPName>` |

Use the prefix to indicate the scope (back‑office or common) followed by the specific section or purpose, then the procedure name.
Sources: [{'source': 'CODINGGUIDELINESFORCTSYS.pdf', 'page': 1, 'score': 0.1930316686630249, 'preview': 'Naming Convention for Stored Procedures :  \n BACKOFFICE_section_SPNAME  \n COMMON_SPName  \n  \n Function Header :  \n /*-------------------------------------------------------------------------------------------------------------------------  \n Function Name:  \n Description: This function checks whethe...'}, {'source': 'CODINGGUIDELINESFORCTSYS.pdf', 'page': 1, 'score': 0.1930316686630249, 'preview': 'Naming Convention for Stored Procedures :  \n BACKOFFICE_section_SPNAME  \n COMMON_SPName  \n  \n Function Header :  \n /*----------------------